# 🔍 Optimización — S4

## Federated Proactive Forest

**Estrategia:** Ranks trees by combined F1 + PCD score. Global aggregation.

**Hiperparámetros:** `alpha_pf`, `t_max`, `f1_weight`, `local_weight`

**Datasets:** Letter, Optdigits, Spambase, Nursery, Sonar, Vowel

**N_CLIENTS:** 3

> ⚡ **Cada celda de dataset es independiente** — ejecuta solo la que necesites.

> 📋 **Rangos leídos en runtime desde** `configs/experiments/optimization/search_spaces.yaml`

In [1]:
# ── Imports & Config ─────────────────────────────────────────────────────
import sys
from pathlib import Path

# Suppress tqdm progress bar warning in Jupyter (ipywidgets not installed)
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm')

ROOT = Path.cwd().parent.parent.parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
import json
import optuna
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import train_test_split

SEED = 42
N_CLIENTS = 3
N_TRIALS = 20
STRATEGY = 'S4'
DATA_DIR = ROOT / 'data'
RESULTS_DIR = ROOT / 'results' / 's6_alpha_optimization'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load search space from YAML (single source of truth)
with open(ROOT / 'configs' / 'experiments' / 'optimization' / 'search_spaces.yaml') as f:
    all_spaces = yaml.safe_load(f)
SPACE = all_spaces[STRATEGY]

from src.domain.dataset.base_adapter import DatasetSplit
from src.application.orchestrators.fl_orchestrator import FLEXOrchestrator

print(f'✅ Project root: {ROOT}')
print(f'✅ Strategy: {STRATEGY}')
print(f'✅ N_CLIENTS: {N_CLIENTS}')
print(f'✅ Search space: {list(SPACE.keys())}')

c:\Users\Adrián Rodríguez\AppData\Local\Programs\Python\Python38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Project root: c:\Users\Adrián Rodríguez\Documents\! Study 📝\📊 KDD 🤖🧠\! Federated learning\federated_proactive_forest
✅ Strategy: S4
✅ N_CLIENTS: 3
✅ Search space: ['alpha_pf', 't_max', 'f1_weight', 'local_weight']


## Dataset: Optdigits

5,620 samples, 64 features (8x8 pixel), 10 classes (0-9), numeric 0-16

In [2]:
# ── Load Optdigits ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'optdigits.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='optdigits')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'optdigits',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_optdigits_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_optdigits_s4_results.json")

📊 Shape: (5620, 65)
✅ Train=4496, Test=1124, Feats=64, Classes=10
🚀 Optimizando S4 en optdigits... (20 trials)


[I 2026-04-06 11:51:49,754] A new study created in memory with name: no-name-e375a28c-d861-4d5a-b139-7fdcbd5376dd



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 48 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 69 árboles entrenados
  TOTAL: 133 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 48/48 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 69/69 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 133 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 48 locales + 85 globales externos (48 propios excluidos) = 133 árboles
  client_1: 16 locales + 117 globales externos (16 propios excluidos) = 133 árboles
  client_2: 69 locales + 64 globales externos (69 propios excluidos) = 133 árboles



[I 2026-04-06 11:53:55,657] Trial 0 finished with value: 0.9688198489707917 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.9688198489707917.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 35 árboles entrenados
  client_1: 100 árboles entrenados
  client_2: 43 árboles entrenados
  TOTAL: 178 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 35/35 árboles seleccionados
  client_1: 100/100 árboles seleccionados
  client_2: 43/43 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 178 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 35 locales + 143 globales externos (35 propios excluidos) = 178 árboles
  client_1: 100 locales + 78 globales externos (100 propios excluidos) = 178 árboles
  client_2: 43 locales + 135 globales externos (43 propios excluidos) = 178 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 76 árboles entrenados
  client_1: 37 árboles entrenados
  client_2: 24 árboles entrenados
  TOTAL: 137 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 76/76 árboles seleccionados
  client_1: 37/37 árboles seleccionados
  client_2: 24/24 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 137 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 76 locales + 61 globales externos (76 propios excluidos) = 137 árboles
  client_1: 37 locales + 100 globales externos (37 propios excluidos) = 137 árboles
  client_2: 24 locales + 113 globales externos (24 propios excluidos) = 137 árboles



[I 2026-04-06 12:43:52,346] Trial 18 finished with value: 0.975084041732752 and parameters: {'alpha_pf': 0.65, 't_max': 120, 'f1_weight': 1.0, 'local_weight': 0.7000000000000001}. Best is trial 18 with value: 0.975084041732752.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 43 árboles entrenados
  client_1: 95 árboles entrenados
  client_2: 46 árboles entrenados
  TOTAL: 184 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 43/43 árboles seleccionados
  client_1: 95/95 árboles seleccionados
  client_2: 46/46 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 184 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 43 locales + 141 globales externos (43 propios excluidos) = 184 árboles
  client_1: 95 locales + 89 globales externos (95 propios excluidos) = 184 árboles
  client_2: 46 locales + 138 globales externos (46 propios excluidos) = 184 árboles



[I 2026-04-06 12:47:08,964] Trial 19 finished with value: 0.9653148637967968 and parameters: {'alpha_pf': 0.7000000000000001, 't_max': 130, 'f1_weight': 1.0, 'local_weight': 0.7000000000000001}. Best is trial 18 with value: 0.975084041732752.



📊 S4 — optdigits — Resultados
   Mejor Macro-F1: 0.9751
   alpha_pf                 : 0.65
   t_max                    : 120
   f1_weight                : 1.0
   local_weight             : 0.7000000000000001
   Media Macro-F1:  0.9686
   Std Macro-F1:    0.0036

✅ Resultados guardados: s6_optdigits_s4_results.json


## Dataset: Spambase

4,601 samples, 57 features (word frequencies), 2 classes (spam/ham)

In [3]:
# ── Load Spambase ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'spambase.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='spambase')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'spambase',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_spambase_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_spambase_s4_results.json")

📊 Shape: (4601, 58)
✅ Train=3680, Test=921, Feats=57, Classes=2
🚀 Optimizando S4 en spambase... (20 trials)


[I 2026-04-06 12:47:09,202] A new study created in memory with name: no-name-a82c4030-cf5a-4385-8ce0-33fb5706960b



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 35 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 78 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 35/35 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 78 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 51 globales externos (27 propios excluidos) = 78 árboles
  client_1: 35 locales + 43 globales externos (35 propios excluidos) = 78 árboles
  client_2: 16 locales + 62 globales externos (16 propios excluidos) = 78 árboles



[I 2026-04-06 12:48:34,871] Trial 0 finished with value: 0.9177490895740099 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.9177490895740099.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 59 árboles entrenados
  client_2: 88 árboles entrenados
  TOTAL: 174 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 59/59 árboles seleccionados
  client_2: 88/88 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 174 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 147 globales externos (27 propios excluidos) = 174 árboles
  client_1: 59 locales + 115 globales externos (59 propios excluidos) = 174 árboles
  client_2: 88 locales + 86 globales externos (88 propios excluidos) = 174 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 46 árboles entrenados
  client_1: 24 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 46/46 árboles seleccionados
  client_1: 24/24 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 46 locales + 40 globales externos (46 propios excluidos) = 86 árboles
  client_1: 24 locales + 62 globales externos (24 propios excluidos) = 86 árboles
  client_2: 16 locales + 70 globales externos (16 propios excluidos) = 86 árboles



[I 2026-04-06 13:28:23,283] Trial 18 finished with value: 0.9291003635496752 and parameters: {'alpha_pf': 0.1, 't_max': 120, 'f1_weight': 0.9, 'local_weight': 0.9}. Best is trial 11 with value: 0.9292444413604418.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 66 árboles entrenados
  client_1: 44 árboles entrenados
  client_2: 100 árboles entrenados
  TOTAL: 210 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 66/66 árboles seleccionados
  client_1: 44/44 árboles seleccionados
  client_2: 100/100 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 210 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 66 locales + 144 globales externos (66 propios excluidos) = 210 árboles
  client_1: 44 locales + 166 globales externos (44 propios excluidos) = 210 árboles
  client_2: 100 locales + 110 globales externos (100 propios excluidos) = 210 árboles



[I 2026-04-06 13:32:36,075] Trial 19 finished with value: 0.9268132785028905 and parameters: {'alpha_pf': 0.1, 't_max': 130, 'f1_weight': 1.0, 'local_weight': 1.0}. Best is trial 11 with value: 0.9292444413604418.



📊 S4 — spambase — Resultados
   Mejor Macro-F1: 0.9292
   alpha_pf                 : 0.25
   t_max                    : 90
   f1_weight                : 0.4
   local_weight             : 0.7000000000000001
   Media Macro-F1:  0.9224
   Std Macro-F1:    0.0039

✅ Resultados guardados: s6_spambase_s4_results.json


## Dataset: Nursery

12,960 samples, 8 categorical features, 5 classes

In [4]:
# ── Load Nursery ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'nursery.csv')
print(f'📊 Shape: {df.shape}')
cat_cols = ['parents', 'has_nurs', 'form', 'children', 'housing', 'finance', 'social', 'health']
enc = OrdinalEncoder()
X = enc.fit_transform(df[cat_cols])
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='nursery')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'nursery',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_nursery_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_nursery_s4_results.json")

📊 Shape: (12960, 9)
✅ Train=10368, Test=2592, Feats=8, Classes=5
🚀 Optimizando S4 en nursery... (20 trials)


[I 2026-04-06 13:32:36,345] A new study created in memory with name: no-name-bf8708e9-8029-4530-808b-0a08ba591594



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 46 árboles entrenados
  TOTAL: 86 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 24/24 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 46/46 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 86 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 62 globales externos (24 propios excluidos) = 86 árboles
  client_1: 16 locales + 70 globales externos (16 propios excluidos) = 86 árboles
  client_2: 46 locales + 40 globales externos (46 propios excluidos) = 86 árboles



[I 2026-04-06 13:33:39,660] Trial 0 finished with value: 0.952553505441209 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.952553505441209.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 84 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 71 árboles entrenados
  TOTAL: 171 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 84/84 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 71/71 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 171 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 84 locales + 87 globales externos (84 propios excluidos) = 171 árboles
  client_1: 16 locales + 155 globales externos (16 propios excluidos) = 171 árboles
  client_2: 71 locales + 100 globales externos (71 propios excluidos) = 171 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 63 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 103 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 24/24 árboles seleccionados
  client_1: 63/63 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 103 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 79 globales externos (24 propios excluidos) = 103 árboles
  client_1: 63 locales + 40 globales externos (63 propios excluidos) = 103 árboles
  client_2: 16 locales + 87 globales externos (16 propios excluidos) = 103 árboles



[I 2026-04-06 13:53:45,681] Trial 18 finished with value: 0.9553786824733523 and parameters: {'alpha_pf': 0.2, 't_max': 70, 'f1_weight': 0.7000000000000001, 'local_weight': 0.1}. Best is trial 15 with value: 0.9648263220020357.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 50 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 82 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 50/50 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 82 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 50 locales + 32 globales externos (50 propios excluidos) = 82 árboles
  client_1: 16 locales + 66 globales externos (16 propios excluidos) = 82 árboles
  client_2: 16 locales + 66 globales externos (16 propios excluidos) = 82 árboles



[I 2026-04-06 13:54:20,211] Trial 19 finished with value: 0.9437756779713821 and parameters: {'alpha_pf': 0.30000000000000004, 't_max': 50, 'f1_weight': 0.9, 'local_weight': 0.4}. Best is trial 15 with value: 0.9648263220020357.



📊 S4 — nursery — Resultados
   Mejor Macro-F1: 0.9648
   alpha_pf                 : 0.15000000000000002
   t_max                    : 70
   f1_weight                : 1.0
   local_weight             : 0.30000000000000004
   Media Macro-F1:  0.9505
   Std Macro-F1:    0.0072

✅ Resultados guardados: s6_nursery_s4_results.json


## Dataset: Sonar

208 samples, 60 numeric features, 2 classes (Rock/Mine) — small dataset!

In [5]:
# ── Load Sonar ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'sonar.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='sonar')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'sonar',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_sonar_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_sonar_s4_results.json")

📊 Shape: (208, 61)
✅ Train=166, Test=42, Feats=60, Classes=2
🚀 Optimizando S4 en sonar... (20 trials)


[I 2026-04-06 13:54:20,317] A new study created in memory with name: no-name-3b635f1b-1b95-4a72-86c5-0474a555ab9d



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 24 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 56 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 24/24 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 56 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 24 locales + 32 globales externos (24 propios excluidos) = 56 árboles
  client_1: 16 locales + 40 globales externos (16 propios excluidos) = 56 árboles
  client_2: 16 locales + 40 globales externos (16 propios excluidos) = 56 árboles



[I 2026-04-06 13:54:24,825] Trial 0 finished with value: 0.8541666666666667 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.8541666666666667.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 35 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 33 árboles entrenados
  TOTAL: 95 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 35/35 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 33/33 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 95 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 35 locales + 60 globales externos (35 propios excluidos) = 95 árboles
  client_1: 27 locales + 68 globales externos (27 propios excluidos) = 95 árboles
  client_2: 33 locales + 62 globales externos (33 propios excluidos) = 95 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 48 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 80 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 48/48 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 80 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 48 locales + 32 globales externos (48 propios excluidos) = 80 árboles
  client_1: 16 locales + 64 globales externos (16 propios excluidos) = 80 árboles
  client_2: 16 locales + 64 globales externos (16 propios excluidos) = 80 árboles



[I 2026-04-06 13:56:34,124] Trial 18 finished with value: 0.7475961538461537 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.6000000000000001, 'local_weight': 0.5}. Best is trial 0 with value: 0.8541666666666667.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 16 árboles entrenados
  client_2: 35 árboles entrenados
  TOTAL: 78 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 16/16 árboles seleccionados
  client_2: 35/35 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 78 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 51 globales externos (27 propios excluidos) = 78 árboles
  client_1: 16 locales + 62 globales externos (16 propios excluidos) = 78 árboles
  client_2: 35 locales + 43 globales externos (35 propios excluidos) = 78 árboles



[I 2026-04-06 13:56:39,758] Trial 19 finished with value: 0.8055555555555555 and parameters: {'alpha_pf': 0.5, 't_max': 140, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.8541666666666667.



📊 S4 — sonar — Resultados
   Mejor Macro-F1: 0.8542
   alpha_pf                 : 0.35
   t_max                    : 150
   f1_weight                : 0.8
   local_weight             : 0.6000000000000001
   Media Macro-F1:  0.8027
   Std Macro-F1:    0.0301

✅ Resultados guardados: s6_sonar_s4_results.json


## Dataset: Vowel

990 samples, 10 numeric features (drop 3 metadata cols), 11 classes

In [6]:
# ── Load Vowel ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'vowel.csv')
print(f'📊 Shape: {df.shape}')
df = df.drop(columns=['Train or Test', 'Speaker Number', 'Sex'])
X = df.drop(columns=['Class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['Class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'Class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='vowel')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'vowel',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_vowel_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_vowel_s4_results.json")

📊 Shape: (990, 14)
✅ Train=792, Test=198, Feats=10, Classes=11
🚀 Optimizando S4 en vowel... (20 trials)


[I 2026-04-06 13:56:39,840] A new study created in memory with name: no-name-08240ec8-1229-46a2-9000-7cfb2dcdeea7



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 100 árboles entrenados
  client_2: 16 árboles entrenados
  TOTAL: 143 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 100/100 árboles seleccionados
  client_2: 16/16 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 143 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 116 globales externos (27 propios excluidos) = 143 árboles
  client_1: 100 locales + 43 globales externos (100 propios excluidos) = 143 árboles
  client_2: 16 locales + 127 globales externos (16 propios excluidos) = 143 árboles



[I 2026-04-06 13:57:16,884] Trial 0 finished with value: 0.7840933384421036 and parameters: {'alpha_pf': 0.35, 't_max': 150, 'f1_weight': 0.8, 'local_weight': 0.6000000000000001}. Best is trial 0 with value: 0.7840933384421036.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 70 árboles entrenados
  client_2: 50 árboles entrenados
  TOTAL: 147 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 70/70 árboles seleccionados
  client_2: 50/50 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 147 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 120 globales externos (27 propios excluidos) = 147 árboles
  client_1: 70 locales + 77 globales externos (70 propios excluidos) = 147 árboles
  client_2: 50 locales + 97 globales externos (50 propios excluidos) = 147 árboles



...


🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 27 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 59 árboles entrenados
  TOTAL: 113 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 27/27 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 59/59 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 113 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 27 locales + 86 globales externos (27 propios excluidos) = 113 árboles
  client_1: 27 locales + 86 globales externos (27 propios excluidos) = 113 árboles
  client_2: 59 locales + 54 globales externos (59 propios excluidos) = 113 árboles



[I 2026-04-06 14:05:10,688] Trial 18 finished with value: 0.7750719115081419 and parameters: {'alpha_pf': 0.7000000000000001, 't_max': 90, 'f1_weight': 0.1, 'local_weight': 1.0}. Best is trial 14 with value: 0.8160172955944135.



🌳 ÁRBOLES ENTRENADOS POR CLIENTE
  client_0: 48 árboles entrenados
  client_1: 27 árboles entrenados
  client_2: 38 árboles entrenados
  TOTAL: 113 árboles

📊 ÁRBOLES SELECCIONADOS (Estrategia: S4)
  client_0: 48/48 árboles seleccionados
  client_1: 27/27 árboles seleccionados
  client_2: 38/38 árboles seleccionados

🌲 BOSQUE GLOBAL
  Tamaño final: 113 árboles

📈 TAMAÑO FINAL POR CLIENTE (sin duplicados)
  client_0: 48 locales + 65 globales externos (48 propios excluidos) = 113 árboles
  client_1: 27 locales + 86 globales externos (27 propios excluidos) = 113 árboles
  client_2: 38 locales + 75 globales externos (38 propios excluidos) = 113 árboles



[I 2026-04-06 14:05:36,220] Trial 19 finished with value: 0.8032235019569668 and parameters: {'alpha_pf': 0.75, 't_max': 100, 'f1_weight': 0.4, 'local_weight': 0.8}. Best is trial 14 with value: 0.8160172955944135.



📊 S4 — vowel — Resultados
   Mejor Macro-F1: 0.8160
   alpha_pf                 : 0.75
   t_max                    : 30
   f1_weight                : 0.30000000000000004
   local_weight             : 0.7000000000000001
   Media Macro-F1:  0.7879
   Std Macro-F1:    0.0189

✅ Resultados guardados: s6_vowel_s4_results.json


## Dataset: Letter

20,000 samples, 16 features, 26 classes (A-Z), numeric features — LARGEST, runs last

In [ ]:
# ── Load Letter ──────────────────────────────────────────────────
df = pd.read_csv(DATA_DIR / 'letter.csv')
print(f'📊 Shape: {df.shape}')
X = df.drop(columns=['class']).values.astype(float)
le = LabelEncoder()
y = le.fit_transform(df['class'])
class_names = list(le.classes_)
feature_names = [c for c in df.columns if c != 'class']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
sc = StandardScaler()
X_tr = sc.fit_transform(X_tr)
X_te = sc.transform(X_te)
ds = DatasetSplit(X_train=X_tr, y_train=y_tr, X_test=X_te, y_test=y_te,
                  feature_names=feature_names, class_names=class_names,
                  dataset_name='letter')
print(f'✅ Train={ds.X_train.shape[0]}, Test={ds.X_test.shape[0]}, Feats={ds.X_train.shape[1]}, Classes={len(class_names)}')

# ── Optimization ──────────────────────────────────────────────────────────
def objective(trial):
    params = {}
    for _name, _cfg in SPACE.items():
        _t = _cfg.get('type', 'float')
        _low, _high = _cfg['low'], _cfg['high']
        _step = _cfg.get('step')
        _log = _cfg.get('log', False)
        if _t == 'float':
            if _log:
                params[_name] = trial.suggest_float(_name, _low, _high, log=True)
            elif _step is not None:
                params[_name] = trial.suggest_float(_name, _low, _high, step=_step)
            else:
                params[_name] = trial.suggest_float(_name, _low, _high)
        elif _t == 'int':
            params[_name] = trial.suggest_int(_name, _low, _high, step=_step or 1)

    agg = {'strategy': STRATEGY}
    if 't_max' in params:
        agg['t_max'] = params['t_max']
    if 'f1_weight' in params:
        agg['f1_weight'] = params['f1_weight']
        agg['pcd_weight'] = 1.0 - params['f1_weight']
    cfg = {
        'federation': {'n_clients': N_CLIENTS, 'distribution': 'iid', 'seed': SEED},
        'model': {
            'n_estimators': 100, 'alpha': params['alpha_pf'],
            'split_criterion': 'entropy', 'use_progressive_stopping': True,
            'convergence': 0.002, 'episode_size': 5, 'verbose': False,
        },
        'aggregation': agg,
        'prediction': {
            'local_weight': params['local_weight'],
            'global_weight': 1.0 - params['local_weight'],
        },
        'verbose': False, 'seed': SEED,
    }
    np.random.seed(SEED)
    orch = FLEXOrchestrator.from_config(cfg)
    orch.setup_federation(ds, seed=SEED)
    res = orch.run_federated_round()
    return res.global_macro_f1

print(f"🚀 Optimizando S4 en {ds.dataset_name}... ({N_TRIALS} trials)")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

df_trials = study.trials_dataframe()
print(f"\n{'='*70}")
print(f"📊 S4 — {ds.dataset_name} — Resultados")
print(f"{'='*70}")
print(f"   Mejor Macro-F1: {study.best_value:.4f}")
for p, v in study.best_params.items():
    print(f"   {p:25s}: {v}")
print(f"   Media Macro-F1:  {df_trials['value'].mean():.4f}")
print(f"   Std Macro-F1:    {df_trials['value'].std():.4f}")

# Save per-dataset results
result = {
    'dataset': 'letter',
    'strategy': 'S4',
    'best_macro_f1': study.best_value,
    'mean_macro_f1': df_trials['value'].mean(),
    'std_macro_f1': df_trials['value'].std(),
    'best_params': study.best_params,
    'all_trials': df_trials.to_dict('records'),
}
with open(RESULTS_DIR / 's6_letter_s4_results.json', 'w') as f:
    json.dump(result, f, indent=2, default=str)
print(f"\n✅ Resultados guardados: s6_letter_s4_results.json")

📊 Shape: (20000, 17)


✅ Train=16000, Test=4000, Feats=16, Classes=26
🚀 Optimizando S4 en letter... (20 trials)


[I 2026-04-06 14:50:45,366] A new study created in memory with name: no-name-baa954b4-479d-4df6-ae3d-bbd1e949614c


## 📊 Resumen Global — Comparar todos los datasets

> ⚡ Ejecuta esta celda **después** de haber ejecutado todas las celdas de datasets.

In [8]:
# ── Resumen Global (ejecutar después de todas las celdas) ─────────────
results = []
for fp in sorted(RESULTS_DIR.glob('s6_*_s4_results.json')):
    with open(fp) as f:
        r = json.load(f)
    row = {'dataset': r['dataset'], 'best_f1': round(r['best_macro_f1'], 4),
           'mean_f1': round(r['mean_macro_f1'], 4), 'std': round(r['std_macro_f1'], 4)}
    row.update(r['best_params'])
    results.append(row)

df_sum = pd.DataFrame(results)
print(f"\n{'='*90}")
print(f"📋 RESUMEN GLOBAL — S4")
print(f"{'='*90}")
if df_sum.empty:
    print('⚠️ No hay resultados. Ejecuta al menos una celda de dataset primero.')
else:
    print(df_sum.to_string(index=False))
    if 'alpha_pf' in df_sum.columns:
        rec = df_sum['alpha_pf'].median()
        print(f"\n🎯 alpha_pf recomendado (mediana): {rec:.1f}")
        print(f"   Rango: [{df_sum['alpha_pf'].min()} — {df_sum['alpha_pf'].max()}]")
    df_sum.to_csv(RESULTS_DIR / 'summary_s4.csv', index=False)
    print(f"\n✅ Resumen guardado: summary_s4.csv")


📋 RESUMEN GLOBAL — S4
  dataset  best_f1  mean_f1    std  alpha_pf  t_max  f1_weight  local_weight
  nursery   0.9648   0.9505 0.0072      0.15     70        1.0           0.3
optdigits   0.9751   0.9686 0.0036      0.65    120        1.0           0.7
    sonar   0.8542   0.8027 0.0301      0.35    150        0.8           0.6
 spambase   0.9292   0.9224 0.0039      0.25     90        0.4           0.7
    vowel   0.8160   0.7879 0.0189      0.75     30        0.3           0.7

🎯 alpha_pf recomendado (mediana): 0.3
   Rango: [0.15000000000000002 — 0.75]

✅ Resumen guardado: summary_s4.csv
